# Pipeline de Modelado para Detección de Vishing — Dataset Aumentado BioCatch (~1M sesiones)

**Iniciativa Anti-Vishing:** Evaluación de modelos base sobre el dataset aumentado sintéticamente.

## Objetivo

Entrenar y evaluar múltiples algoritmos de clasificación (Regresión Logística, Random Forest, XGBoost y Redes Neuronales) iterando sobre los diferentes datasets generados en el proceso de balanceo de datos de la data aumentada. Todo esto usando como base las variables generadas para el dataset aumentado.

**Diferencias clave vs pipeline original:**
- Imbalance severo base: 1:94 (vs 1:19 en el dataset original)
- Sin scores BioCatch (no disponibles en el dataset aumentado)

**Manejo del desbalance:**
- Iteración sobre variantes de balanceo dentro de `data/balanced/augmented/`

**Métrica prioritaria:** PR-AUC y Recall (contexto de detección de fraude con imbalance 1:94)

## 1. Setup e Importaciones
Para evaluar de forma justa, extraemos un set de prueba del dataset aumentado sin balancear, tal como se hizo en el pipeline original.


In [9]:
import pandas as pd
import numpy as np
import os
import glob
from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import (
    recall_score, precision_score, f1_score,
    roc_auc_score, average_precision_score,
    confusion_matrix, precision_recall_curve
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_sample_weight

import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print('Librerías cargadas correctamente.')


Librerías cargadas correctamente.


## 2. Carga del Dataset Aumentado

El dataset `data\augmented_data\dataset_1M_vishing_.parquet.parquet` contiene ~1M sesiones sinteticas generadas a partir del dataset original de 50K sesiones.


In [10]:
PARQUET_PATH = 'data/augmented_data/dataset_1M_vishing_.parquet'

df = pd.read_parquet(PARQUET_PATH)
df['session_timestamp'] = pd.to_datetime(df['session_timestamp'])

n_total   = len(df)
n_vishing = int((df['is_vishing'] == 1).sum())
n_legit   = int((df['is_vishing'] == 0).sum())
imbalance_ratio = n_legit // n_vishing

print(f'Dataset aumentado : {n_total:,} filas x {df.shape[1]} columnas')
print(f'  Legítimas       : {n_legit:,}  ({n_legit/n_total*100:.2f}%)')
print(f'  Vishing         : {n_vishing:,}   ({n_vishing/n_total*100:.2f}%)')
print(f'  Imbalance ratio : 1:{imbalance_ratio}')

# ── Extract Hold-out: 20% estratificado (virgen) ──
cols_to_drop = ['session_id', 'customer_id', 'session_timestamp',
           'device_type', 'os_type', 'app_version',
           'biocatch_risk_score', 'biocatch_genuine_score', 'biocatch_ato_indicator', 
           'biocatch_social_eng_indicator', 'biocatch_bot_indicator', 'days_to_claim', 'claim_category',
           'screens_visited', 'unusual_screen_visits']
           
df_model = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

X = df_model.drop(columns=['is_vishing'])
y = df_model['is_vishing']

X_train_pool, X_test, y_train_pool, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print('\nHOLD-OUT SET (20% del total — virgen)')
print(f'  Total     : {len(X_test):,}')
print(f'  Legítimas : {(y_test==0).sum():,} ({(y_test==0).mean()*100:.2f}%)')
print(f'  Vishing   : {y_test.sum():,} ({y_test.mean()*100:.2f}%)')

scaler = StandardScaler()

Dataset aumentado : 1,000,000 filas x 62 columnas
  Legítimas       : 985,000  (98.50%)
  Vishing         : 15,000   (1.50%)
  Imbalance ratio : 1:65

HOLD-OUT SET (20% del total — virgen)
  Total     : 200,000
  Legítimas : 197,000 (98.50%)
  Vishing   : 3,000 (1.50%)


## 3. Definición del Pipeline Multimodelo

Iteraremos a través de cada carpeta y archivo Parquet generado en el proceso de balanceo dentro de `data/balanced/augmented/`.


In [11]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=150, max_depth=10, random_state=42, n_jobs=-1),
    'XGBoost': XGBClassifier(use_label_encoder=False, eval_metric='logloss', max_depth=6, learning_rate=0.1, random_state=42, n_jobs=-1),
    'Deep Learning (MLP)': MLPClassifier(hidden_layer_sizes=(64, 32), activation='relu', solver='adam', max_iter=300, random_state=42)
}

data_paths = glob.glob('data/balanced/augmented/**/*.parquet', recursive=True)
results = []

print(f"Detectados {len(data_paths)} datasets balanceados para evaluación.")

Detectados 12 datasets balanceados para evaluación.


In [ ]:
import joblib
import os

for path in data_paths:
    technique = Path(path).parent.name
    ratio = Path(path).stem
    
    print(f"\n--- Evaluando {technique} al {ratio}% ---")
    
    # Crear directorio para guardar los modelos
    model_dir = Path(f"modelos/{technique}/{ratio}")
    model_dir.mkdir(parents=True, exist_ok=True)
    
    df_train = pd.read_parquet(path)
    df_train = df_train.drop(columns=[c for c in cols_to_drop if c in df_train.columns], errors='ignore')
    
    X_train = df_train.drop(columns=['is_vishing'])
    y_train = df_train['is_vishing']
    
    X_train = X_train[X_test.columns]
    
    scaler.fit(X_train)
    X_train_scaled = scaler.transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    for model_name, model in models.items():
        if model_name in ['Logistic Regression', 'Deep Learning (MLP)']:
            X_tr, X_te = X_train_scaled, X_test_scaled
        else:
            X_tr, X_te = X_train.values, X_test.values
            
        model.fit(X_tr, y_train)
        
        # Guardar el modelo entrenado
        safe_model_name = model_name.replace(' ', '_').replace('(', '').replace(')', '')
        model_path = model_dir / f"{safe_model_name}.pkl"
        joblib.dump(model, model_path)
        
        y_pred = model.predict(X_te)
        y_prob = model.predict_proba(X_te)[:, 1] if hasattr(model, 'predict_proba') else y_pred
        
        recall = recall_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        roc_auc = roc_auc_score(y_test, y_prob)
        pr_auc = average_precision_score(y_test, y_prob)
        
        results.append({
            'Technique': technique,
            'Ratio_%': ratio,
            'Model': model_name,
            'Recall': recall,
            'Precision': precision,
            'F1': f1,
            'ROC_AUC': roc_auc,
            'PR_AUC': pr_auc
        })
        
print("\n¡Pipeline de iteración masiva finalizado exitosamente! Modelos guardados en ./modelos/")


--- Evaluando borderline_smote al 10% ---

--- Evaluando borderline_smote al 20% ---

--- Evaluando borderline_smote al 25% ---


## 4. Análisis de Resultados e Identificación de la Mejor Estrategia


In [ ]:
df_results = pd.DataFrame(results)

df_results_sorted = df_results.sort_values(by='PR_AUC', ascending=False)
display(df_results_sorted.head(15).style.background_gradient(cmap='viridis', subset=['Recall', 'PR_AUC', 'ROC_AUC']))


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))

sns.barplot(data=df_results, x='Technique', y='PR_AUC', hue='Model', ax=ax1)
ax1.set_title('Comparación de PR-AUC por Técnica y Modelo')
ax1.set_xticklabels(ax1.get_xticklabels(), rotation=45)

sns.barplot(data=df_results, x='Ratio_%', y='Recall', hue='Model', ax=ax2)
ax2.set_title('Impacto del Ratio de Balanceo sobre el Recall')

plt.tight_layout()
plt.show()


## 5. Analisis del Mejor Modelo (Matriz de Confusion)

Seleccionaremos la combinación de Técnica, Ratio y Modelo que obtuvo el mejor PR-AUC y trazaremos su matriz de confusión sobre el Hold-Out Set.


In [ ]:
import joblib

best_row = df_results_sorted.iloc[0]
best_technique = best_row['Technique']
best_ratio = best_row['Ratio_%']
best_model_name = best_row['Model']

print(f"--- Mejor combinación identificada (por PR-AUC) ---")
print(f"Modelo  : {best_model_name}")
print(f"Técnica : {best_technique}")
print(f"Ratio   : {best_ratio}%")
print(f"PR_AUC  : {best_row['PR_AUC']:.4f}")

# --- Fallback: Cargar modelo si no está en memoria ---
safe_model_name = best_model_name.replace(' ', '_').replace('(', '').replace(')', '')
model_path = Path(f"modelos/{best_technique}/{best_ratio}/{safe_model_name}.pkl")

if 'models' in locals() and best_model_name in models:
    best_model = models[best_model_name]
    print("\nModelo encontrado en memoria. Usando instancia existente.")
else:
    try:
        best_model = joblib.load(model_path)
        print(f"\nModelo '{best_model_name}' cargado desde: {model_path}")
    except FileNotFoundError:
        print(f"Error: No se encontró el archivo del modelo en {model_path}. Re-entrenando...")
        # Re-entrenar como fallback si el archivo no existe
        best_model = RandomForestClassifier(n_estimators=150, max_depth=10, random_state=42, n_jobs=-1) # Usar un modelo base o el correcto
        if best_model_name == 'XGBoost':
            best_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', max_depth=6, learning_rate=0.1, random_state=42, n_jobs=-1)
        # Añadir lógica para otros modelos si es necesario...

# Re-entrenar o cargar datos para la evaluación
best_path = f'data/balanced/augmented/{best_technique}/{best_ratio}.parquet'
df_best = pd.read_parquet(best_path)

X_best = df_best.drop(columns=['is_vishing'])[X_test.columns]
y_best = df_best['is_vishing']

scaler_best = StandardScaler()
scaler_best.fit(X_best)

use_scaled = best_model_name in ['Logistic Regression', 'Deep Learning (MLP)']
X_tr_best = scaler_best.transform(X_best) if use_scaled else X_best.values
X_te_best = scaler_best.transform(X_test) if use_scaled else X_test.values

# Si el modelo no fue cargado, se entrena
if not ('best_model' in locals() and hasattr(best_model, 'predict')):
    best_model.fit(X_tr_best, y_best)

y_pred_best = best_model.predict(X_te_best)
y_prob_best = best_model.predict_proba(X_te_best)[:, 1] if hasattr(best_model, 'predict_proba') else y_pred_best

cm = confusion_matrix(y_test, y_pred_best)
tn, fp, fn, tp = cm.ravel()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Matriz de confusión con conteos
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
    xticklabels=['Legítimo (pred)', 'Vishing (pred)'],
    yticklabels=['Legítimo (real)', 'Vishing (real)']
)
axes[0].set_title('Matriz de Confusión — Conteos', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Clase Real')
axes[0].set_xlabel('Clase Predicha')

# Matriz de confusión normalizada (por fila = recall por clase)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
sns.heatmap(
    cm_norm, annot=True, fmt='.2%', cmap='Blues', ax=axes[1],
    xticklabels=['Legítimo (pred)', 'Vishing (pred)'],
    yticklabels=['Legítimo (real)', 'Vishing (real)']
)
axes[1].set_title('Matriz de Confusión — Normalizada (Recall por clase)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Clase Real')
axes[1].set_xlabel('Clase Predicha')

plt.suptitle(f'{best_model_name} · {best_technique} {best_ratio}% · Hold-out set', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print(f"\n{'='*50}")
print(f"  Verdaderos Negativos (TN) : {tn:>6}  — Legítimos correctamente identificados")
print(f"  Falsos Positivos     (FP) : {fp:>6}  — Legítimos marcados como Vishing")
print(f"  Falsos Negativos     (FN) : {fn:>6}  — Vishing no detectados  ← riesgo crítico")
print(f"  Verdaderos Positivos (TP) : {tp:>6}  — Vishing correctamente detectados")
print(f"{'='*50}")
print(f"  Recall    (Sensibilidad)  : {tp/(tp+fn):.4f}")
print(f"  Precision                 : {tp/(tp+fp):.4f}")
print(f"  F1-Score                  : {2*tp/(2*tp+fp+fn):.4f}")
print(f"  ROC-AUC                   : {roc_auc_score(y_test, y_prob_best):.4f}")
print(f"  PR-AUC                    : {average_precision_score(y_test, y_prob_best):.4f}")
print(f"{'='*50}")

## 6. Feature Importance — XGBoost

XGBoost ofrece interpretabilidad nativa via importancia de variables por Gain acumulado.


In [ ]:
if 'XGBoost' in models:
    xgb_model = models['XGBoost']
    # If the model wasn't trained in the last cell due to not being the best, we must fit it on some dataset. 
    # For now, let's use the 'best' dataset to see the feature importances on the best balanced strategy.
    
    xgb_model.fit(X_best.values, y_best) # X_best and y_best were loaded in previous cell

    FEATURE_NAMES = X_test.columns.tolist()

    fi = pd.DataFrame({
        'feature'   : FEATURE_NAMES,
        'importance': xgb_model.feature_importances_
    }).sort_values('importance', ascending=False).reset_index(drop=True)

    fig, ax = plt.subplots(figsize=(12, 10))
    top_fi = fi.head(25)

    q80 = top_fi['importance'].quantile(0.80)
    q60 = top_fi['importance'].quantile(0.60)
    colors_fi = ['#e74c3c' if v >= q80 else '#f39c12' if v >= q60 else '#3498db'
                 for v in top_fi['importance']]

    bars = ax.barh(range(len(top_fi)), top_fi['importance'].values,
                   color=colors_fi, edgecolor='white', linewidth=0.5)
    ax.set_yticks(range(len(top_fi)))
    ax.set_yticklabels(top_fi['feature'].values)
    ax.set_xlabel('Importancia (Gain acumulado)')
    ax.set_title('Feature Importance — XGBoost  |  Top 25 Features', fontweight='bold', fontsize=14)
    ax.invert_yaxis()
    for i, v in enumerate(top_fi['importance'].values):
        ax.text(v + top_fi['importance'].max()*0.005, i, f'{v:.4f}', va='center', fontsize=8)

    from matplotlib.patches import Patch
    ax.legend(handles=[
        Patch(facecolor='#e74c3c', label='Top 20% importancia'),
        Patch(facecolor='#f39c12', label='Top 20-40%'),
        Patch(facecolor='#3498db', label='Resto'),
    ], loc='lower right')

    plt.tight_layout()
    plt.show()
    
    print('Top 15 features (XGBoost — Gain):')
    print(f"{'Feature':40s} {'Importancia':>12s}")
    print('-' * 55)
    for _, row in fi.head(15).iterrows():
        print(f"{row['feature']:40s} {row['importance']:12.4f}")
else:
    print("XGBoost no está en el diccionario de modelos, no se grafican importancias.")


## 7. Curva Precision-Recall y Analisis de Umbral de Decision

In [ ]:
precisions_c, recalls_c, thresholds_c = precision_recall_curve(y_test, y_prob_best)
pr_auc_val = average_precision_score(y_test, y_prob_best)

f1_scores_c = (2 * precisions_c[:-1] * recalls_c[:-1]
               / (precisions_c[:-1] + recalls_c[:-1] + 1e-9))
best_idx  = np.argmax(f1_scores_c)
best_thr  = thresholds_c[best_idx]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Curva PR
axes[0].plot(recalls_c, precisions_c, color='#3498db', linewidth=2,
             label=f'PR-AUC = {pr_auc_val:.4f}')
axes[0].axhline(y=y_test.mean(), color='gray', linestyle='--', alpha=0.7,
                label=f'Baseline sin modelo ({y_test.mean():.4f})')
axes[0].fill_between(recalls_c, precisions_c, alpha=0.08, color='#3498db')
axes[0].scatter([recalls_c[best_idx]], [precisions_c[best_idx]], s=100,
                color='red', zorder=5, label=f'Óptimo F1 (thr={best_thr:.3f})')
axes[0].set_xlabel('Recall (Sensibilidad)', fontsize=12)
axes[0].set_ylabel('Precision', fontsize=12)
axes[0].set_title(f'Curva Precision-Recall — {best_model_name}', fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].set_xlim(0, 1)
axes[0].set_ylim(0, 1.05)

# Metricas vs umbral
axes[1].plot(thresholds_c, precisions_c[:-1], label='Precision', color='#2ecc71', linewidth=2)
axes[1].plot(thresholds_c, recalls_c[:-1],    label='Recall',    color='#e74c3c', linewidth=2)
axes[1].plot(thresholds_c, f1_scores_c,       label='F1',        color='#f39c12', linewidth=2, linestyle='--')
axes[1].axvline(x=best_thr, color='black', linestyle=':', alpha=0.8,
                label=f'Óptimo F1 = {best_thr:.3f}')
axes[1].axvline(x=0.5, color='gray', linestyle=':', alpha=0.5, label='Default (0.5)')
axes[1].set_xlabel('Umbral de Decisión', fontsize=12)
axes[1].set_ylabel('Métrica', fontsize=12)
axes[1].set_title('Precision, Recall y F1 vs Umbral', fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].set_xlim(0, 1)
axes[1].set_ylim(0, 1.05)

plt.suptitle(f'Ánalisis del Umbral de Decisión — {best_model_name}',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

y_pred_opt = (y_prob_best >= best_thr).astype(int)

print(f"{'='*62}")
print(f"  Con umbral óptimo (max F1) = {best_thr:.4f}")
print(f"    Recall    : {recall_score(y_test, y_pred_opt):.4f}")
print(f"    Precision : {precision_score(y_test, y_pred_opt):.4f}")
print(f"    F1        : {f1_score(y_test, y_pred_opt):.4f}")
print(f"{'='*62}")
print(f"  Con umbral por defecto = 0.5")
print(f"    Recall    : {recall_score(y_test, y_pred_best):.4f}")
print(f"    Precision : {precision_score(y_test, y_pred_best):.4f}")
print(f"    F1        : {f1_score(y_test, y_pred_best):.4f}")
print(f"{'='*62}")
